# XGBoost - Cross Validation (UCI-THYROID-DXBIN)

Notebook para executar XGBoost com StratifiedKFold sobre o dataset UCI-THYROID-DXBIN.
- Selecione qual dataset normalizado usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `xgb_params`.
- Outputs: `model_reports/xgboost/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/UCI-THYROID-DXBIN/common/models/xgboost/v1/xgb_model_cf{N_SPLITS}.pkl`.

In [ ]:
# Configuração (XGBoost pronto p/ 3 classes)
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário

# Dataset (ajuste para o seu multi-classe)
DATASET_NAME = 'uci_thyroid_dxbin_processed.csv'  # troque se tiver versão multi-classe
DATASET_PATH = Path('../data/processed') / DATASET_NAME

# Coluna-alvo (deve conter {0,1,2} para 3 classes)
TARGET_COLUMN = 'y'  # ajustar se necessário

# Saídas
REPORTS = Path('../model_reports/xgboost/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/xgboost/v1')
# Básico (single train/test) - pasta irmã de 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# === Número de classes ===
# Se quiser detectar depois de carregar o dataset, você pode sobrescrever N_CLASSES lá.
N_CLASSES = 3  # <- 3 classes por padrão

# XGBoost params (multi-classe por padrão)
xgb_params = {
    "n_estimators": 650,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_child_weight": 1,
    # ------ objetivos/métricas multi vs binário ------
    "objective": "multi:softprob" if N_CLASSES > 2 else "binary:logistic",
    "num_class": N_CLASSES if N_CLASSES > 2 else None,
    "eval_metric": "mlogloss" if N_CLASSES > 2 else "logloss",
    # -------------------------------------------------
    "tree_method": "hist",
    "n_jobs": 8,
    "random_state": 42
}
# Remove a chave None para binário (evita warning do XGB)
if xgb_params["num_class"] is None:
    xgb_params.pop("num_class")

# CV & decisão
N_SPLITS = 10
# Para multi-classe usamos argmax; para binário você pode definir THRESHOLD=0.5
THRESHOLD = None if N_CLASSES > 2 else 0.5
DECISION_RULE = 'argmax' if N_CLASSES > 2 else 'threshold'  # usado na etapa de predição/avaliação

# Colunas a excluir do treinamento (mantidas nas saídas)f
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', '_target', 'target','target_']

# Discretização (se aplicável ao seu pipeline)
VARIABLE_DISCRETIZE = [
    'sex','on_thyroxine','query_on_thyroxine','on_antithyroid_meds','sick','pregnant',
    'thyroid_surgery','I131_treatment','query_hypothyroid','query_hyperthyroid','lithium',
    'goitre','tumor','hypopituitary','psych','TSH_measured','TSH','T3_measured','T3',
    'TT4_measured','TT4','T4U_measured','T4U','FTI_measured','FTI','TBG_measured'
]
VARIABLE_DISCRETIZE_NUMBER = {'other':0, 'SVI':1, 'SVHC':2, 'STMW':3, 'SVHD':4, 'WEST':5}

# Pareto (feature-importance acumulada)
PARETO_THRESHOLD = 0.9
PARETO_SELECTED_FEATURES = None  # será preenchido após CV

# Diferença percentual de variância (para checagens de drift / consistência)
VAR_DIFF_PCT_THRESHOLD = 0.01

# Holdout básico (caso use também split simples além de CV)
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42



In [ ]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [ ]:
# Load dataset (path definido na configuração) — pronto para 3 classes
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserva cópia bruta do CSV original
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leitura
df = pd.read_csv(DATASET_PATH)

# Util: localizar coluna (case/strip tolerant)
def find_column(df, name):
    if name in df.columns:
        return name
    low = {c.lower().strip(): c for c in df.columns}
    key = name.lower().strip()
    if key in low:
        return low[key]
    for c in df.columns:
        if key in c.lower():
            return c
    return None

# Determina coluna-alvo
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    target_col = None
    for cand in ['y', 'diagnosis', 'target', 'label', 'class', 'diagnose']:
        c = find_column(df, cand)
        if c:
            target_col = c; break
    if target_col is None:
        # tenta binária
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c; break
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia alvo para 'y'
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Discretiza/normaliza rótulos para inteiros 0..K-1
class_mapping_path = CSV_OUT / f'class_mapping_{DATASET_NAME.replace(".csv","")}.csv'
y_original = df['y'].copy()

if not np.issubdtype(df['y'].dtype, np.number):
    # Ex.: 'M'/'B' → binário; outros → fatoriza multi-classe
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= {'m', 'b'}:
        mapping = {v: (1 if str(v).lower() == 'm' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
        pd.DataFrame({'original': list(mapping.keys()), 'encoded': list(mapping.values())}).to_csv(class_mapping_path, index=False)
    else:
        df['y'], uniques = pd.factorize(df['y'])
        pd.DataFrame({'original': uniques.astype(str), 'encoded': range(len(uniques))}).to_csv(class_mapping_path, index=False)
else:
    # Numérico: se não estiver em 0..K-1, reindexa de forma estável
    y_vals = df['y'].dropna().unique()
    y_vals_sorted = np.sort(y_vals)
    if not np.array_equal(y_vals_sorted, np.arange(y_vals_sorted.size)):
        remap = {orig: i for i, orig in enumerate(y_vals_sorted)}
        df['y'] = df['y'].map(remap)
        pd.DataFrame({'original': y_vals_sorted, 'encoded': [remap[v] for v in y_vals_sorted]}).to_csv(class_mapping_path, index=False)
    else:
        # Mesmo assim, salve o mapping identidade
        pd.DataFrame({'original': y_vals_sorted, 'encoded': y_vals_sorted}).to_csv(class_mapping_path, index=False)

# Sanitiza: inf/NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# Garante 'y' por último
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# Salva dataset usado
used_path = CSV_OUT / f'database_used_{DATASET_NAME}'
if used_path.suffix.lower() != '.csv':
    used_path = used_path.with_suffix('.csv')
df.to_csv(used_path, index=False)

# Detecta nº de classes e ajusta params globais p/ XGBoost
N_CLASSES = int(df['y'].nunique())
globals()['N_CLASSES'] = N_CLASSES

xgb_params = globals().get('xgb_params', {})
if N_CLASSES > 2:
    xgb_params.update({
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'num_class': N_CLASSES
    })
    globals()['THRESHOLD'] = None
    globals()['DECISION_RULE'] = 'argmax'
else:
    xgb_params.update({
        'objective': 'binary:logistic',
        'eval_metric': 'logloss'
    })
    xgb_params.pop('num_class', None)
    globals()['THRESHOLD'] = 0.5
    globals()['DECISION_RULE'] = 'threshold'

globals()['xgb_params'] = xgb_params

print('Dataset shape:', df.shape)
print('N_CLASSES:', N_CLASSES)
print('xgb_params:', xgb_params)
print('CSV usado:', used_path)
print('Mapping salvo em:', class_mapping_path)


In [ ]:
# Repetir o codigo abaixo e analisar o porque esta demoando muito ... 

In [ ]:
# ===================== OPTUNA (TPE + PRUNER) PARA XGBOOST — 8/1/1 (compat legado) =====================
import numpy as np
import pandas as pd
from pathlib import Path
import optuna
from optuna.exceptions import TrialPruned
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, accuracy_score,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
import xgboost as xgb  # callbacks, se disponível

# ------------------------- PRÉ-CHECAGENS -------------------------
try:
    df
except NameError:
    raise RuntimeError("df não está definido.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', 'fold', 'y_pred', 'y_proba', 'prob_0', 'prob_1']

if "y" not in df.columns:
    raise RuntimeError("Coluna 'y' não encontrada em df.")

# ------------------------- CONFIGS -------------------------
K = 3
RANDOM_SEED = 42

N_TRIALS_OPTUNA   = int(globals().get("N_TRIALS_OPTUNA", 20))
N_SPLITS_OUTER    = 10        # 8/1/1 fixo (fold 9 val, fold 10 test)
N_SPLITS_INNER    = 8         # optuna CV interna
EARLY_STOP_ROUNDS = 50

EPS = 1e-12

# >>>>>>> DEFINIÇÃO DO "POSITIVO" PARA TP/FP/TN/FN (one-vs-rest) <<<<<<<<
POS_CLASS = int(globals().get("POS_CLASS", 1))  # escolha: 0, 1 ou 2
if POS_CLASS not in [0, 1, 2]:
    raise ValueError("POS_CLASS deve ser 0, 1 ou 2 para K=3.")

# Threshold tuning (para POS_CLASS)
THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.01))
THRESH_MIN = float(globals().get("THRESH_MIN", 0.01))
THRESH_MAX = float(globals().get("THRESH_MAX", 0.99))

# evita threshold “esperto” que prediz zero positivos
MIN_POS_PRED = globals().get("MIN_POS_PRED", 1)  # pode ser int (>=1) ou fração (0..1)

# pasta de saída
OUT_DIR = Path('../model_reports/xgboost/cv/csv')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INNER_FOLDS = OUT_DIR / "cv_811_inner_folds.csv"
CSV_VAL         = OUT_DIR / "cv_811_validation.csv"
CSV_TEST        = OUT_DIR / "cv_811_test.csv"
CSV_RESULTS     = OUT_DIR / "cv_811_results_long.csv"
CSV_THR         = OUT_DIR / f"best_threshold_cv811_posclass{POS_CLASS}_f1.json"

# ------------------------- Monta X / y -------------------------
y_local = df["y"].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
id_cols = [c for c in df.columns if c.lower() in exclude_lower and c != "y"]
FEATURES_OPT = [c for c in df.columns if c not in id_cols + ["y"]]

X_local = df[FEATURES_OPT].copy()
X_local.replace([np.inf, -np.inf], np.nan, inplace=True)
X_local = X_local.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# garante rótulos 0..K-1
classes = np.sort(pd.unique(y_local.dropna()))
if len(classes) != K:
    raise RuntimeError(f"Esperado K={K} classes, mas encontrei {len(classes)}: {classes}")

if not np.array_equal(classes, np.arange(K)):
    mapping = {c: i for i, c in enumerate(classes)}
    y_local = y_local.map(mapping).astype(int)
else:
    y_local = y_local.astype(int)

print(f"✅ X_local={X_local.shape} | y={y_local.shape} | classes={np.sort(y_local.unique())}")
print(f"🧷 POS_CLASS (one-vs-rest p/ TP/FP/TN/FN): {POS_CLASS}")

# ======================================================================
# ------------------------ HELPERS: CE, AUC, TP/FP/TN/FN OVR ------------
# ======================================================================
def _cross_entropy_per_class_multiclass(y_true, proba_all, n_classes=3, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    P = np.clip(np.asarray(proba_all), eps, 1.0 - eps)
    out = {}
    for k in range(n_classes):
        m = (y_true == k)
        out[f"CE-C{k}"] = float(np.mean(-np.log(P[m, k]))) if np.any(m) else np.nan
    return out

def _safe_auc_ovr_macro(y_true, proba_all):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y_true, proba_all, multi_class="ovr", average="macro"))
    except Exception:
        return np.nan

def _counts_binary(y_true_bin, y_pred_bin):
    y_true_bin = np.asarray(y_true_bin).astype(int)
    y_pred_bin = np.asarray(y_pred_bin).astype(int)
    TP = int(np.sum((y_true_bin == 1) & (y_pred_bin == 1)))
    TN = int(np.sum((y_true_bin == 0) & (y_pred_bin == 0)))
    FP = int(np.sum((y_true_bin == 0) & (y_pred_bin == 1)))
    FN = int(np.sum((y_true_bin == 1) & (y_pred_bin == 0)))
    total = TP + TN + FP + FN
    return TP, FP, TN, FN, total

def _min_pos_as_int(n, min_pos_pred):
    # aceita int ou fração
    if isinstance(min_pos_pred, (float, np.floating)) and 0 < float(min_pos_pred) < 1:
        return max(1, int(np.ceil(float(min_pos_pred) * n)))
    try:
        return max(1, int(min_pos_pred))
    except Exception:
        return 1

def find_best_threshold_posclass_f1(y_true_mc, p_pos, step=0.01, tmin=0.01, tmax=0.99, min_pos_pred=1):
    """
    Otimiza F1 one-vs-rest da POS_CLASS na validação.
    Evita threshold que prediz 0 positivos (controlado por MIN_POS_PRED).
    """
    y_true_mc = np.asarray(y_true_mc).astype(int)
    p_pos = np.asarray(p_pos)

    y_true_bin = (y_true_mc == POS_CLASS).astype(int)
    n = len(y_true_bin)
    min_pos_int = _min_pos_as_int(n, min_pos_pred)

    best_t = 0.5
    best_s = -1.0
    candidates = 0

    for t in np.arange(tmin, tmax + 1e-12, step):
        y_pred_bin = (p_pos >= t).astype(int)
        if int(y_pred_bin.sum()) < min_pos_int:
            continue
        s = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))
        candidates += 1
        if s > best_s:
            best_s = s
            best_t = float(t)

    # fallback: se nada passou no min_pos, relaxa a regra
    if candidates == 0:
        for t in np.arange(tmin, tmax + 1e-12, step):
            y_pred_bin = (p_pos >= t).astype(int)
            s = float(f1_score(y_true_bin, y_pred_bin, zero_division=0))
            if s > best_s:
                best_s = s
                best_t = float(t)
        return best_t, best_s, 0, True, min_pos_int

    return best_t, best_s, candidates, False, min_pos_int

def _metrics_row_multiclass_legacy(etapa, fold, y_true, proba_all, thr_pos):
    """
    - Métricas multiclasse (macro): ACC, F1, BAC, PREC, REC
    - AUROC: OVR macro (fallback BAC se falhar)
    - CE-C0/1/2: por classe (média -log p_classe_real)
    - TP/FP/TN/FN: definidos via one-vs-rest da POS_CLASS usando threshold thr_pos
    """
    y_true = np.asarray(y_true).astype(int)
    P = np.asarray(proba_all)
    y_pred_argmax = np.argmax(P, axis=1)

    # macro
    acc  = float(accuracy_score(y_true, y_pred_argmax))
    f1m  = float(f1_score(y_true, y_pred_argmax, average="macro", zero_division=0))
    bacc = float(balanced_accuracy_score(y_true, y_pred_argmax))
    prec = float(precision_score(y_true, y_pred_argmax, average="macro", zero_division=0))
    rec  = float(recall_score(y_true, y_pred_argmax, average="macro", zero_division=0))

    # proporções (fração)
    prop0 = float(np.mean(y_true == 0))
    prop1 = float(np.mean(y_true == 1))

    # CE por classe
    ce = _cross_entropy_per_class_multiclass(y_true, P, n_classes=K, eps=EPS)

    # AUROC OVR macro (fallback)
    auc_val = _safe_auc_ovr_macro(y_true, P)
    if np.isnan(auc_val):
        auc_val = bacc

    # one-vs-rest para POS_CLASS
    p_pos = P[:, POS_CLASS]
    y_true_bin = (y_true == POS_CLASS).astype(int)
    y_pred_bin = (p_pos >= thr_pos).astype(int)
    TP, FP, TN, FN, total = _counts_binary(y_true_bin, y_pred_bin)

    TPp = TP / total if total > 0 else np.nan
    FPp = FP / total if total > 0 else np.nan
    TNp = TN / total if total > 0 else np.nan
    FNp = FN / total if total > 0 else np.nan

    return {
        "etapa": etapa,
        "fold": int(fold),

        "Acurácia": acc,
        "Cross-Entropy C0": ce["CE-C0"],
        "Proporção C0": prop0,
        "Cross-Entropy C1": ce["CE-C1"],
        "Proporção C1": prop1,

        "F1 Score": f1m,
        "Balanced Accuracy": bacc,
        "Precision": prec,
        "Recall": rec,

        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TPp, "FP(%)": FPp, "TN(%)": TNp, "FN(%)": FNp,

        "AUROC": auc_val,
        "Cross-Entropy C2": ce["CE-C2"],

        # extra pra auditoria
        "POS_CLASS": POS_CLASS,
        "thr_posclass": float(thr_pos),
    }

# ======================================================================
# -------------------------- SPLIT 8/1/1 HOLD-OUT FIXO ------------------
# ======================================================================
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_SEED)
folds = list(outer_kf.split(X_local, y_local))

val_idx  = folds[-2][1]  # fold 9
test_idx = folds[-1][1]  # fold 10
train_idx = np.setdiff1d(np.arange(len(X_local)), np.concatenate([val_idx, test_idx]))

X_train8, y_train8 = X_local.iloc[train_idx], y_local.iloc[train_idx]
X_val1,   y_val1   = X_local.iloc[val_idx],   y_local.iloc[val_idx]
X_test1,  y_test1  = X_local.iloc[test_idx],  y_local.iloc[test_idx]

print("\n=== Split 8/1/1 (hold-out) — MULTICLASSE ===")
print(f"Treino: {len(train_idx)} | Validação hold-out: {len(val_idx)} | Teste hold-out: {len(test_idx)}")

# ======================================================================
# -------------------------- OPTUNA: CV interna (8 folds no TREINO) ------
# ======================================================================
inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_SEED)

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 400, 2000, step=100),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_weight":  trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma":             trial.suggest_float("gamma", 1e-8, 10.0, log=True),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "tree_method":       "hist",
        "max_bin":           trial.suggest_int("max_bin", 128, 512, step=64),
        "grow_policy":       trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "random_state":      RANDOM_SEED,
        "n_jobs":            -1,
        "objective":         "multi:softprob",
        "eval_metric":       "mlogloss",
        "num_class":         K,
        "verbosity":         0,
    }

    scores = []
    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, y_tr = X_train8.iloc[tr_idx], y_train8.iloc[tr_idx]
        X_va, y_va = X_train8.iloc[va_idx], y_train8.iloc[va_idx]

        model = XGBClassifier(**params)

        callbacks = []
        try:
            callbacks = [xgb.callback.EarlyStopping(rounds=EARLY_STOP_ROUNDS, save_best=True, maximize=False)]
        except Exception:
            callbacks = []

        try:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False, callbacks=callbacks)
        except TypeError:
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        p_va = model.predict_proba(X_va)

        s = _safe_auc_ovr_macro(y_va, p_va)
        if np.isnan(s):
            s = float(balanced_accuracy_score(y_va, np.argmax(p_va, axis=1)))

        scores.append(float(s))
        trial.report(float(np.mean(scores)), step=fold_id)
        if trial.should_prune():
            raise TrialPruned()

    return float(np.mean(scores))

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED, multivariate=True, n_startup_trials=10)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="xgb_811_optuna_multiclass_holdout_posclass",
)

print("\n🔧 Iniciando Optuna (multiclasse) …")
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print("\n✅ Melhor valor (score médio CV interna):", study.best_value)
print("✅ Melhores hiperparâmetros:", study.best_params)

_best = study.best_params.copy()
xgb_params = {
    "n_jobs": -1,
    "random_state": RANDOM_SEED,
    "tree_method": "hist",
    "verbosity": 0,
    **_best,
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "num_class": K,
}

print("\n📦 xgb_params finais:\n", xgb_params)

# ======================================================================
# -------------------------- (A) treina no treino8 e calibra threshold (fold 9) p/ POS_CLASS ----
# ======================================================================
clf_thr = XGBClassifier(**xgb_params)
clf_thr.fit(X_train8, y_train8)
p_val_all = clf_thr.predict_proba(X_val1)
p_val_pos = p_val_all[:, POS_CLASS]

best_thr, best_f1, n_cand, fb, min_pos_int = find_best_threshold_posclass_f1(
    y_true_mc=y_val1,
    p_pos=p_val_pos,
    step=THRESH_GRID_STEP,
    tmin=THRESH_MIN,
    tmax=THRESH_MAX,
    min_pos_pred=MIN_POS_PRED,
)

print("\n[DEBUG] Probabilidades POS_CLASS na validação hold-out:")
print(f"min={p_val_pos.min():.6f} | p50={np.percentile(p_val_pos,50):.6f} | p90={np.percentile(p_val_pos,90):.6f} | max={p_val_pos.max():.6f}")
print(f"🎚️ Threshold ótimo (POS_CLASS={POS_CLASS}, F1 OVR): {best_thr:.3f} | F1={best_f1:.6f} | candidatos={n_cand} | fallback={fb} | min_pos={min_pos_int}")

# salva threshold
import json
with open(CSV_THR, "w") as f:
    json.dump(
        {
            "POS_CLASS": POS_CLASS,
            "best_threshold": best_thr,
            "best_f1_posclass": best_f1,
            "grid_step": THRESH_GRID_STEP,
            "tmin": THRESH_MIN,
            "tmax": THRESH_MAX,
            "MIN_POS_PRED": MIN_POS_PRED,
            "min_pos_int": min_pos_int,
            "fallback_used": bool(fb),
        },
        f,
        indent=2
    )

# ======================================================================
# ===================== (B) AVALIAÇÃO: inner_cv (8 folds) + val hold-out + teste hold-out
# ======================================================================
metrics_rows = []

# (1) inner_cv no TREINO (8 folds), usando threshold calibrado (best_thr) para preencher TP/FP/TN/FN
for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
    X_tr, y_tr = X_train8.iloc[tr_idx], y_train8.iloc[tr_idx]
    X_va, y_va = X_train8.iloc[va_idx], y_train8.iloc[va_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr)
    p_va_all = model.predict_proba(X_va)

    metrics_rows.append(_metrics_row_multiclass_legacy("inner_cv", fold_id, y_va, p_va_all, thr_pos=best_thr))

# (2) validação hold-out (fold 9)
metrics_rows.append(_metrics_row_multiclass_legacy("validacao", 9, y_val1, p_val_all, thr_pos=best_thr))

# (3) treino final (8+1) e teste hold-out (fold 10)
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

clf_final = XGBClassifier(**xgb_params)
clf_final.fit(X_train9, y_train9)

p_test_all = clf_final.predict_proba(X_test1)
p_test_pos = p_test_all[:, POS_CLASS]

print("\n[DEBUG] Probabilidades POS_CLASS no teste hold-out:")
print(f"min={p_test_pos.min():.6f} | p50={np.percentile(p_test_pos,50):.6f} | p90={np.percentile(p_test_pos,90):.6f} | max={p_test_pos.max():.6f}")

metrics_rows.append(_metrics_row_multiclass_legacy("teste", 10, y_test1, p_test_all, thr_pos=best_thr))

df_metrics = pd.DataFrame(metrics_rows)

# ======================================================================
# -------------------- SALVAR CSVs --------------------
# ======================================================================
df_metrics.to_csv(CSV_RESULTS, index=False)
df_metrics[df_metrics["etapa"] == "inner_cv"].to_csv(CSV_INNER_FOLDS, index=False)
df_metrics[df_metrics["etapa"] == "validacao"].to_csv(CSV_VAL, index=False)
df_metrics[df_metrics["etapa"] == "teste"].to_csv(CSV_TEST, index=False)

pd.options.display.float_format = lambda x: f"{x:.6f}"

print("\n===== Resultados (LONG) =====")
print(df_metrics.to_string(index=False))

print("\n===== Resumo por etapa (média) =====")
print(df_metrics.groupby("etapa").mean(numeric_only=True).round(6))

print("\n📝 Arquivos salvos:")
print(" -", CSV_INNER_FOLDS)
print(" -", CSV_VAL)
print(" -", CSV_TEST)
print(" -", CSV_RESULTS)
print(" -", CSV_THR)
print("✔️ Agora TP/FP/TN/FN NÃO são NaN: são OVR para POS_CLASS com threshold calibrado na validação.")


In [ ]:
import pandas as pd
from pathlib import Path

CSV_RESULTS = Path(r"../UCI-THYROID-DXBIN/model_reports/xgboost/cv/csv/cv_811_results_long.csv")
df_metrics = pd.read_csv(CSV_RESULTS)
print(df_metrics.columns.tolist())
print(df_metrics.head(2))


In [ ]:
# ============================================================
# OVR por classe (POS_CLASS = 0,1,2) com threshold por classe
# - escolhe thr na VALIDACAO (maximize F1 por classe)
# - aplica no TESTE (mesmo thr da validação daquela classe)
# - salva tabela + CSV
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    accuracy_score, balanced_accuracy_score, roc_auc_score
)

# ------------------ CONFIGS ------------------
K = int(globals().get("K", 3))  # classes 0..K-1
THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.01))
THRESH_MIN = float(globals().get("THRESH_MIN", 0.01))
THRESH_MAX = float(globals().get("THRESH_MAX", 0.99))

# Robustez: evita escolher threshold que prevê "quase ninguém positivo"
# pode ser fração (0..1) ou inteiro (>=1)
MIN_POS_PRED = globals().get("MIN_POS_PRED", 1)  # ex: 0.01 (=1%) ou 5 (5 amostras)

# Saída
CSV_OUT = globals().get("CSV_OUT", None)
OUT_DIR = globals().get("OUT_DIR", None)

if CSV_OUT is None:
    if OUT_DIR is not None:
        CSV_OUT = OUT_DIR
    else:
        CSV_OUT = Path(".")
CSV_OUT = Path(CSV_OUT)
CSV_OUT.mkdir(parents=True, exist_ok=True)

OUT_CSV_OVR = CSV_OUT / "cv_811_ovr_posclass_thresholds.csv"

# ------------------ Helpers ------------------
def _as_int_labels(y):
    y = pd.Series(y).astype(int)
    return y.to_numpy()

def _safe_auc_bin(y_bin, p):
    # só calcula se existir 0 e 1 presentes
    u = np.unique(y_bin)
    if len(u) < 2:
        return np.nan
    try:
        return float(roc_auc_score(y_bin, p))
    except Exception:
        return np.nan

def _min_pos_to_int(min_pos_pred, n):
    # se vier como fração (0<min<1), converte para contagem mínima
    if isinstance(min_pos_pred, (float, np.floating)) and (0 < float(min_pos_pred) < 1.0):
        return int(np.ceil(float(min_pos_pred) * n))
    try:
        v = int(min_pos_pred)
        return max(v, 0)
    except Exception:
        return 0

def _conf_counts(y_true_bin, y_pred_bin):
    y_true_bin = np.asarray(y_true_bin).astype(int)
    y_pred_bin = np.asarray(y_pred_bin).astype(int)

    TP = int(np.sum((y_true_bin == 1) & (y_pred_bin == 1)))
    TN = int(np.sum((y_true_bin == 0) & (y_pred_bin == 0)))
    FP = int(np.sum((y_true_bin == 0) & (y_pred_bin == 1)))
    FN = int(np.sum((y_true_bin == 1) & (y_pred_bin == 0)))
    total = TP + TN + FP + FN

    return TP, FP, TN, FN, total

def _metrics_ovr(y_true_bin, p_pos, thr):
    y_true_bin = np.asarray(y_true_bin).astype(int)
    p_pos = np.asarray(p_pos)
    y_pred = (p_pos >= thr).astype(int)

    TP, FP, TN, FN, total = _conf_counts(y_true_bin, y_pred)

    acc  = float(accuracy_score(y_true_bin, y_pred))
    f1   = float(f1_score(y_true_bin, y_pred, zero_division=0))
    prec = float(precision_score(y_true_bin, y_pred, zero_division=0))
    rec  = float(recall_score(y_true_bin, y_pred, zero_division=0))
    bac  = float(balanced_accuracy_score(y_true_bin, y_pred))

    auc  = _safe_auc_bin(y_true_bin, p_pos)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "ACC_bin": acc,
        "F1_bin": f1,
        "BAC_bin": bac,
        "PRE_bin": prec,
        "REC_bin": rec,
        "AUC_bin": auc,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
        "N_pos_true": int(np.sum(y_true_bin == 1)),
        "N_neg_true": int(np.sum(y_true_bin == 0)),
        "N_pos_pred": int(np.sum(y_pred == 1)),
    }

def _choose_threshold_for_posclass(y_true, proba_all, pos_class, step=0.01,
                                  tmin=0.01, tmax=0.99,
                                  optimize_for="f1",
                                  min_pos_pred=1):
    """
    Escolhe threshold na validação para a classe pos_class vs resto.
    optimize_for: "f1" (padrão), "bac" ou "precision" ou "recall"
    """
    y_true = _as_int_labels(y_true)
    P = np.asarray(proba_all)
    p_pos = P[:, pos_class]
    y_bin = (y_true == pos_class).astype(int)

    n = len(y_bin)
    min_pos = _min_pos_to_int(min_pos_pred, n)

    best_thr = 0.5
    best_score = -np.inf
    best_valid = False
    candidates = 0

    for thr in np.arange(tmin, tmax + 1e-12, step):
        y_pred = (p_pos >= thr).astype(int)
        if int(np.sum(y_pred == 1)) < min_pos:
            continue  # não aceita threshold que prevê poucos positivos

        candidates += 1

        if optimize_for == "bac":
            s = float(balanced_accuracy_score(y_bin, y_pred))
        elif optimize_for == "precision":
            s = float(precision_score(y_bin, y_pred, zero_division=0))
        elif optimize_for == "recall":
            s = float(recall_score(y_bin, y_pred, zero_division=0))
        else:
            s = float(f1_score(y_bin, y_pred, zero_division=0))

        if s > best_score:
            best_score = s
            best_thr = float(thr)
            best_valid = True

    # fallback se não achou nenhum threshold que respeite min_pos
    if not best_valid:
        # escolhe o menor thr para garantir pelo menos min_pos (se possível)
        order = np.argsort(-p_pos)  # desc
        if min_pos <= n and min_pos >= 1:
            kth = p_pos[order[min_pos - 1]]
            best_thr = float(kth)
        else:
            best_thr = 0.5
        # score do fallback
        y_pred = (p_pos >= best_thr).astype(int)
        if optimize_for == "bac":
            best_score = float(balanced_accuracy_score(y_bin, y_pred))
        elif optimize_for == "precision":
            best_score = float(precision_score(y_bin, y_pred, zero_division=0))
        elif optimize_for == "recall":
            best_score = float(recall_score(y_bin, y_pred, zero_division=0))
        else:
            best_score = float(f1_score(y_bin, y_pred, zero_division=0))

    return best_thr, best_score, candidates, best_valid

# ------------------ Obtém probabilidades (val/test) ------------------
# Espera existir: X_val1, y_val1, X_test1, y_test1
for v in ["X_val1", "y_val1", "X_test1", "y_test1"]:
    if v not in globals():
        raise RuntimeError(f"Variável obrigatória ausente: {v}")

X_val1  = globals()["X_val1"]
y_val1  = globals()["y_val1"]
X_test1 = globals()["X_test1"]
y_test1 = globals()["y_test1"]

# Espera existir clf_val e clf_final (treinados no seu pipeline)
if "clf_val" not in globals():
    raise RuntimeError("Não encontrei clf_val treinado (modelo treinado no treino8 para prever validação).")
if "clf_final" not in globals():
    raise RuntimeError("Não encontrei clf_final treinado (modelo treinado no treino9 para prever teste).")

clf_val   = globals()["clf_val"]
clf_final = globals()["clf_final"]

P_val  = clf_val.predict_proba(X_val1)
P_test = clf_final.predict_proba(X_test1)

# ------------------ Loop POS_CLASS 0..K-1 ------------------
OPT_FOR = globals().get("THRESH_OPTIMIZE_FOR", "f1").lower().strip()
if OPT_FOR not in {"f1", "bac", "precision", "recall"}:
    OPT_FOR = "f1"

rows = []

for pos_class in range(K):
    # escolhe threshold no VAL para esta classe
    thr, thr_score, candidates, ok = _choose_threshold_for_posclass(
        y_true=y_val1,
        proba_all=P_val,
        pos_class=pos_class,
        step=THRESH_GRID_STEP,
        tmin=THRESH_MIN,
        tmax=THRESH_MAX,
        optimize_for=OPT_FOR,
        min_pos_pred=MIN_POS_PRED
    )

    # métricas OVR na validação
    y_val_bin = (_as_int_labels(y_val1) == pos_class).astype(int)
    p_val_pos = np.asarray(P_val)[:, pos_class]
    m_val = _metrics_ovr(y_val_bin, p_val_pos, thr)

    rows.append({
        "split": "validacao",
        "POS_CLASS": pos_class,
        "thr_posclass": thr,
        "thr_score": float(thr_score),
        "thr_metric": OPT_FOR,
        "thr_candidates": int(candidates),
        "thr_ok": bool(ok),
        **m_val
    })

    # métricas OVR no teste (mesmo threshold da validação)
    y_te_bin = (_as_int_labels(y_test1) == pos_class).astype(int)
    p_te_pos = np.asarray(P_test)[:, pos_class]
    m_te = _metrics_ovr(y_te_bin, p_te_pos, thr)

    rows.append({
        "split": "teste",
        "POS_CLASS": pos_class,
        "thr_posclass": thr,
        "thr_score": float(thr_score),  # score do threshold no VAL (referência)
        "thr_metric": OPT_FOR,
        "thr_candidates": int(candidates),
        "thr_ok": bool(ok),
        **m_te
    })

df_ovr = pd.DataFrame(rows)

# ordena para ficar bonito
cols_first = [
    "split","POS_CLASS","thr_posclass","thr_metric","thr_score","thr_candidates","thr_ok",
    "ACC_bin","F1_bin","BAC_bin","PRE_bin","REC_bin","AUC_bin",
    "TP","FP","TN","FN","TP(%)","FP(%)","TN(%)","FN(%)",
    "N_pos_true","N_neg_true","N_pos_pred"
]
cols = [c for c in cols_first if c in df_ovr.columns] + [c for c in df_ovr.columns if c not in cols_first]
df_ovr = df_ovr[cols]

# print tabela
pd.options.display.float_format = lambda x: f"{x:.6f}" if isinstance(x, (float, np.floating)) else str(x)

print("\n=== OVR POR CLASSE (threshold por POS_CLASS ajustado na validação) ===")
print(df_ovr.to_string(index=False))

# salva
df_ovr.to_csv(OUT_CSV_OVR, index=False)
print(f"\n✅ CSV OVR salvo em: {OUT_CSV_OVR.resolve()}")


In [ ]:
xgb_params 

In [ ]:
# ================== XGBoost 8/1/1 (alteração mínima, mantendo sua lógica) ==================
import time, joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_curve, roc_auc_score, accuracy_score,
    f1_score, balanced_accuracy_score, precision_score, recall_score, log_loss
)
from matplotlib.backends.backend_pdf import PdfPages
from xgboost import XGBClassifier

# ------------------ y e detecção de IDs/FEATURES ------------------
y = df['y']

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower and c != 'y']
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]

# base para modelagem (sem IDs)
X_model = df[FEATURES].copy()

# ------------------ limpeza e codificação robusta (como no seu bloco) ------------------
X_model.replace([np.inf, -np.inf], np.nan, inplace=True)

num_cols = X_model.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_model.columns if c not in num_cols]

def _maybe_binary_map(series: pd.Series):
    mapping_sets = [
        {'t':1, 'f':0},
        {'true':1, 'false':0},
        {'yes':1, 'no':0},
        {'y':1, 'n':0},
        {'m':1, 'b':0},
        {'male':1, 'female':0},
        {'m':1, 'f':0},
        {'1':1, '0':0}
    ]
    vals = series.dropna().astype(str).str.strip().str.lower().unique().tolist()
    if len(vals) == 2:
        s = set(vals)
        for mp in mapping_sets:
            if s <= set(mp.keys()):
                return series.astype(str).str.strip().str.lower().map(mp).astype('float')
    return None

for c in cat_cols:
    mapped = _maybe_binary_map(X_model[c])
    if mapped is not None:
        X_model[c] = mapped
    else:
        codes = X_model[c].astype('category').cat.codes
        codes = codes.replace({-1: np.nan})
        X_model[c] = codes.astype('float')

for c in X_model.columns:
    col = X_model[c]
    if col.notna().any():
        X_model[c] = col.fillna(col.median(skipna=True))
    else:
        X_model[c] = col.fillna(0.0)

X_model = X_model.apply(pd.to_numeric, errors='coerce').fillna(0.0)

# ------------------ 8/1/1 split ------------------
# 10% TESTE direto
X_rem, X_test, y_rem, y_test = train_test_split(
    X_model, y, test_size=0.10, stratify=y, random_state=42
)

# Dos 90% restantes, separa 10% do total para VALIDAÇÃO -> 0.10/0.90 = 1/9
X_train, X_val, y_train, y_val = train_test_split(
    X_rem, y_rem, test_size=(1/9), stratify=y_rem, random_state=42
)

# Índices originais p/ juntar IDs/merge depois
train_idx = X_train.index
val_idx   = X_val.index
test_idx  = X_test.index

feats_cols_out = ID_COLS + FEATURES if ID_COLS else FEATURES
X_train_full = df.loc[train_idx, feats_cols_out]
X_val_full   = df.loc[val_idx,   feats_cols_out]
X_test_full  = df.loc[test_idx,  feats_cols_out]

# ------------------ estruturas (mantidas) ------------------
resultados = []
importancias_lista = []

X_train_parts, X_val_parts, X_test_parts = [], [], []
y_train_parts, y_val_parts, y_test_parts = [], [], []
y_train_pred_parts, y_val_pred_parts, y_test_pred_parts = [], [], []

K = int(df['y'].nunique())
prob_cols = [f'prob_{k}' for k in range(K)]

fprs_train, tprs_train, aurocs_train = [], [], []
fprs_test,  tprs_test,  aurocs_test  = [], [], []

start_total = time.time()
fold = 1  # único "fold" para manter compatibilidade

pdf_name = PDF_OUT / f'XGB_CV_811.pdf'  # nome separado para 8/1/1; mude se quiser manter o antigo
with PdfPages(pdf_name) as pdf:
    with tqdm(total=1, desc="🔄 Split 8/1/1") as pbar:
        t0 = time.time()

        # ------------------ ajusta objetivo XGB p/ binário vs multi ------------------
        params = dict(xgb_params)  # cópia
        if K == 2:
            params.setdefault("objective", "binary:logistic")
            params.setdefault("eval_metric", "logloss")
        else:
            params["objective"] = "multi:softprob"
            params["eval_metric"] = "mlogloss"
            params["num_class"] = K

        # ------------------ MODELO INNER (80%) -> Treino / Validação ------------------
        model_inner = XGBClassifier(**params)
        model_inner.fit(X_train, y_train)

        # Probabilidades (n x K) para Treino e Validação (modelo inner)
        y_proba_train_all = model_inner.predict_proba(X_train)
        y_proba_val_all   = model_inner.predict_proba(X_val)

        # Predição hard para Treino / Validação
        if K == 2:
            y_proba_train_C1 = y_proba_train_all[:, 1]
            y_proba_val_C1   = y_proba_val_all[:, 1]
            y_pred_train = (y_proba_train_C1 >= THRESHOLD).astype(int)
            y_pred_val   = (y_proba_val_C1   >= THRESHOLD).astype(int)
        else:
            y_pred_train = np.argmax(y_proba_train_all, axis=1)
            y_pred_val   = np.argmax(y_proba_val_all,   axis=1)

        # ------------------ MODELO FINAL (90% = Treino + Validação) -> Teste ------------------
        X_final = pd.concat([X_train, X_val])
        y_final = pd.concat([y_train, y_val])

        model_final = XGBClassifier(**params)
        model_final.fit(X_final, y_final)

        # Probabilidades para TESTE usando o modelo FINAL (90%)
        y_proba_test_all = model_final.predict_proba(X_test)

        if K == 2:
            y_proba_test_C1 = y_proba_test_all[:, 1]
            y_pred_test  = (y_proba_test_C1  >= THRESHOLD).astype(int)
        else:
            y_pred_test  = np.argmax(y_proba_test_all,  axis=1)


        # ------- armazenar blocos com índice original (treino/val/test) -------
        X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
        y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
        y_train_pred_parts.append(
            pd.DataFrame(y_proba_train_all, columns=prob_cols, index=X_train_full.index)
              .assign(fold=fold).reset_index()
        )

        X_val_parts.append(X_val_full.assign(fold=fold).reset_index())
        y_val_parts.append(pd.Series(y_val, name='y_val').to_frame().assign(fold=fold).reset_index())
        y_val_pred_parts.append(
            pd.DataFrame(y_proba_val_all, columns=prob_cols, index=X_val_full.index)
              .assign(fold=fold).reset_index()
        )

        X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
        y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
        y_test_pred_parts.append(
            pd.DataFrame(y_proba_test_all, columns=prob_cols, index=X_test_full.index)
              .assign(fold=fold).reset_index()
        )

        # ===================== Métricas (Treino / Validação / Teste) =====================
        def _add_result(conjunto, y_true, y_pred, proba_all, proba_pos=None):
            if K == 2:
                cm = confusion_matrix(y_true, y_pred, labels=[0,1])
                tn, fp, fn, tp = cm.ravel()
                total = cm.sum()
                if proba_pos is None:
                    proba_pos = proba_all[:, 1]
                try:
                    auc_val = roc_auc_score(y_true, proba_pos)
                except Exception:
                    auc_val = np.nan
                mask0 = (y_true == 0); mask1 = (y_true == 1)
                ll0 = log_loss(y_true[mask0], proba_pos[mask0], labels=[0,1]) if mask0.sum()>0 else np.nan
                ll1 = log_loss(y_true[mask1], proba_pos[mask1], labels=[0,1]) if mask1.sum()>0 else np.nan
                prop = pd.Series(y_true).value_counts(normalize=True).reindex(range(K), fill_value=0) * 100
                resultados.append({
                    'Conjunto': conjunto,
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_true, y_pred),
                    'Cross-Entropy C0': ll0,
                    'Proporção C0': prop.get(0, np.nan),
                    'Cross-Entropy C1': ll1,
                    'Proporção C1': prop.get(1, np.nan),
                    'F1 Score': f1_score(y_true, y_pred),
                    'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
                    'Precision': precision_score(y_true, y_pred),
                    'Recall': recall_score(y_true, y_pred),
                    'AUROC': auc_val,
                    'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
                    'TP(%)': round(tp/total*100, 2),
                    'FP(%)': round(fp/total*100, 2),
                    'TN(%)': round(tn/total*100, 2),
                    'FN(%)': round(fn/total*100, 2),
                    'Tempo (s)': round(time.time() - t0, 2)
                })
                return auc_val
            else:
                cm = confusion_matrix(y_true, y_pred, labels=list(range(K)))
                total = cm.sum()
                try:
                    auc_val = roc_auc_score(y_true, proba_all, multi_class='ovr', average='macro')
                except Exception:
                    auc_val = np.nan
                prop = pd.Series(y_true).value_counts(normalize=True).reindex(range(K), fill_value=0) * 100
                resultados.append({
                    'Conjunto': conjunto,
                    'Fold': fold,
                    'Acurácia': accuracy_score(y_true, y_pred),
                    'Cross-Entropy C0': np.nan,
                    'Proporção C0': prop.get(0, np.nan),
                    'Cross-Entropy C1': np.nan,
                    'Proporção C1': prop.get(1, np.nan),
                    'F1 Score': f1_score(y_true, y_pred, average='macro'),
                    'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
                    'Precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
                    'Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
                    'AUROC': auc_val,
                    'TP': np.nan, 'FP': np.nan, 'TN': np.nan, 'FN': np.nan,
                    'TP(%)': np.nan, 'FP(%)': np.nan, 'TN(%)': np.nan, 'FN(%)': np.nan,
                    'Tempo (s)': round(time.time() - t0, 2)
                })
                return auc_val

        # Treino
        _ = _add_result('Treino', y_train, y_pred_train, y_proba_train_all,
                        (y_proba_train_all[:,1] if K==2 else None))
        # Validação
        _ = _add_result('Validação', y_val, y_pred_val, y_proba_val_all,
                        (y_proba_val_all[:,1] if K==2 else None))
        # Teste
        _ = _add_result('Teste', y_test, y_pred_test, y_proba_test_all,
                        (y_proba_test_all[:,1] if K==2 else None))

        # Curvas ROC (apenas para binário) — mantém suas listas
        if K == 2:
            fpr_tr, tpr_tr, _ = roc_curve(y_train, y_proba_train_C1)
            fpr_ts, tpr_ts, _ = roc_curve(y_test,  y_proba_test_C1)
            fprs_train.append(fpr_tr); tprs_train.append(tpr_tr)
            fprs_test.append(fpr_ts);  tprs_test.append(tpr_ts)
            aurocs_train.append(roc_auc_score(y_train, y_proba_train_C1))
            aurocs_test.append(roc_auc_score(y_test,  y_proba_test_C1))

        # Importância das features (percentual) — usa o modelo FINAL (90%)
        importancias = model_final.feature_importances_
        linha_importancia = {'Conjunto': 'Teste', 'Fold': fold}
        for nome, valor in zip(FEATURES, importancias):
            linha_importancia[nome] = f"{valor * 100:.2f}%"
        importancias_lista.append(linha_importancia)

        # salva modelo FINAL por "fold"
        model_path = MODEL_DIR / f'xgb_model_811_fold{fold}.pkl'
        joblib.dump(model_final, model_path)


        pbar.update(1)

# ------------------ Concatena e salva resultados ------------------
df_resultados = pd.DataFrame(resultados)
num_cols_res = df_resultados.select_dtypes(include=[np.number]).columns
mean_row = df_resultados[num_cols_res].mean(numeric_only=True)
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# (mantendo nomes padrão; troque por *_811.csv se preferir)
df_resultados.to_csv(CSV_OUT / f'xgb_cv_results_{N_SPLITS}.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'xgb_feature_importances_{N_SPLITS}.csv', index=False)

# Blocos completos com probabilidades (inclui validação)
def _concat_or_empty(lst):
    return pd.concat(lst, ignore_index=True) if lst else pd.DataFrame()

X_train_global = _concat_or_empty(X_train_parts)
y_train_global = _concat_or_empty(y_train_parts)
y_train_pred_global = _concat_or_empty(y_train_pred_parts)

X_val_global = _concat_or_empty(X_val_parts)
y_val_global = _concat_or_empty(y_val_parts)
y_val_pred_global = _concat_or_empty(y_val_pred_parts)

X_test_global = _concat_or_empty(X_test_parts)
y_test_global = _concat_or_empty(y_test_parts)
y_test_pred_global = _concat_or_empty(y_test_pred_parts)

for df_block in [X_train_global, X_val_global, X_test_global,
                 y_train_global, y_val_global, y_test_global,
                 y_train_pred_global, y_val_pred_global, y_test_pred_global]:
    if not df_block.empty and 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mesclas finais (train/val/test)
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_val_global.empty and not y_val_global.empty:
    val_df = X_val_global.merge(y_val_global, on=['orig_index', 'fold'], how='left')
    if not y_val_pred_global.empty:
        val_df = val_df.merge(y_val_pred_global, on=['orig_index', 'fold'], how='left')
else:
    val_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()

# Saídas finais (binário vs multi) — mantém a lógica
if not test_df.empty:
    if K == 2:
        test_df['y_proba'] = test_df['prob_1']
        test_df['y_pred']  = (test_df['y_proba'] >= THRESHOLD).astype(int)
    else:
        present = [c for c in prob_cols if c in test_df.columns]
        test_df['y_proba_max'] = test_df[present].max(axis=1)
        test_df['y_pred'] = test_df[present].idxmax(axis=1).str.replace('prob_', '').astype(int)
    test_df.to_csv(CSV_OUT / f'xgb_test_with_proba_{N_SPLITS}.csv', index=False)

if not train_df.empty:
    if K == 2:
        train_df['y_proba'] = train_df['prob_1']
        train_df['y_pred']  = (train_df['y_proba'] >= THRESHOLD).astype(int)
    else:
        present = [c for c in prob_cols if c in train_df.columns]
        train_df['y_proba_max'] = train_df[present].max(axis=1)
        train_df['y_pred'] = train_df[present].idxmax(axis=1).str.replace('prob_', '').astype(int)
    train_df.to_csv(CSV_OUT / f'xgb_train_with_proba_{N_SPLITS}.csv', index=False)

# ➕ Validação
if not val_df.empty:
    if K == 2:
        val_df['y_proba'] = val_df['prob_1']
        val_df['y_pred']  = (val_df['y_proba'] >= THRESHOLD).astype(int)
    else:
        present = [c for c in prob_cols if c in val_df.columns]
        val_df['y_proba_max'] = val_df[present].max(axis=1)
        val_df['y_pred'] = val_df[present].idxmax(axis=1).str.replace('prob_', '').astype(int)
    val_df.to_csv(CSV_OUT / f'xgb_val_with_proba_{N_SPLITS}.csv', index=False)

# Dataset augmentado (usa TESTE)
try:
    if not test_df.empty:
        df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
        if K == 2:
            preds = test_df[['orig_index', 'y_pred', 'y_proba']]
        else:
            preds = test_df[['orig_index', 'y_pred', 'y_proba_max']].rename(columns={'y_proba_max':'y_proba'})
        df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
        df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_xgb_{N_SPLITS}.csv', index=False)
    else:
        print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
except Exception as e_aug:
    print('Erro ao gerar dataset augmentado com previsões:', e_aug)

print('✅ Split 8/1/1 concluído. Resultados salvos em:', CSV_OUT)
print('📄 PDF gerado em:', pdf_name)


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# Geração de PDF resumo (multi-classe pronto: 2 ou 3+ classes) — versão 8/1/1 com Validação opcional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

# ===== helpers numéricos para variância/violinos =====
def _is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)

def _coerce_numeric_df(df_in: pd.DataFrame, cols):
    """Tenta converter cols para numérico (errors='coerce'); remove colunas 100% NaN após a coerção."""
    if df_in is None or len(cols) == 0:
        return pd.DataFrame()
    out = pd.DataFrame(index=df_in.index)
    for c in cols:
        if c in df_in.columns:
            out[c] = pd.to_numeric(df_in[c], errors='coerce')
    keep = [c for c in out.columns if not out[c].isna().all()]
    return out[keep]

def _numeric_common_features(train_df: pd.DataFrame, test_df: pd.DataFrame, candidates):
    """Retorna lista de cols numéricas comuns (ou convertíveis) em train e test."""
    if train_df.empty or test_df.empty:
        return []
    num_native = [c for c in candidates if c in train_df.columns and c in test_df.columns
                  and _is_numeric_series(train_df[c]) and _is_numeric_series(test_df[c])]
    if len(num_native) > 0:
        return num_native
    cand = [c for c in candidates if c in train_df.columns and c in test_df.columns]
    tnc = _coerce_numeric_df(train_df, cand)
    tec = _coerce_numeric_df(test_df, cand)
    common = [c for c in tnc.columns if c in tec.columns]
    good = []
    for c in common:
        if tnc[c].notna().any() and tec[c].notna().any():
            good.append(c)
    return good

# ===== número de classes =====
try:
    K = int(df['y'].nunique())
except Exception:
    if 'test_df' in globals() and isinstance(test_df, pd.DataFrame):
        prob_cols = [c for c in test_df.columns if c.startswith('prob_')]
        K = max(int(c.split('_')[1]) for c in prob_cols) + 1 if prob_cols else 2
    else:
        K = 2
class_labels = list(range(K))

with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # Página de parâmetros
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    parametros['num_class_detectado'] = K
    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k,v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig); plt.close()

    # Página de métricas resumidas (médias)
    if 'df_resultados' in globals() and not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False); table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig); plt.close()

    # ---------------- ROC (Treino/Validação/Teste) ----------------
    def _macro_roc_from_df(df_block, y_true_col, prob_cols_prefix='prob_'):
        if df_block.empty: return None
        prob_cols = [c for c in df_block.columns if c.startswith(prob_cols_prefix)]
        if not prob_cols:
            return None
        prob_cols = sorted(prob_cols, key=lambda s: int(s.split('_')[1]))  # prob_0..prob_{K-1}
        y_true = df_block[y_true_col].astype(int).values
        K_local = len(prob_cols)
        Y = label_binarize(y_true, classes=list(range(K_local)))
        mean_fpr = np.linspace(0, 1, 200)
        tprs, aucs = [], []
        scores = df_block[prob_cols].values
        for k in range(K_local):
            fpr_k, tpr_k, _ = roc_curve(Y[:, k], scores[:, k])
            tpr_i = np.interp(mean_fpr, fpr_k, tpr_k); tpr_i[0] = 0.0
            tprs.append(tpr_i)
            try:
                aucs.append(auc(fpr_k, tpr_k))
            except Exception:
                pass
        if not tprs: return None
        mean_tpr = np.mean(np.array(tprs), axis=0); mean_tpr[-1] = 1.0
        auc_macro = np.nanmean(aucs) if len(aucs) else auc(mean_fpr, mean_tpr)
        return mean_fpr, mean_tpr, auc_macro

    try:
        plotted_any = False

        # Detecta presença dos dataframes
        _train_df = globals().get('train_df', pd.DataFrame())
        _val_df   = globals().get('val_df',   pd.DataFrame())
        _test_df  = globals().get('test_df',  pd.DataFrame())

        have_train = (isinstance(_train_df, pd.DataFrame) and not _train_df.empty and 'y_train' in _train_df.columns)
        have_val   = (isinstance(_val_df,   pd.DataFrame) and not _val_df.empty   and 'y_val'   in _val_df.columns)
        have_test  = (isinstance(_test_df,  pd.DataFrame) and not _test_df.empty  and 'y_test'  in _test_df.columns)

        roc_parts = []  # lista de tuplas (titulo, fpr, tpr, auc_val)

        # Treino
        if have_train:
            if K == 2 and 'prob_1' in _train_df.columns:
                y_tr, ysc_tr = _train_df['y_train'], _train_df['prob_1']
                fpr_tr, tpr_tr, _ = roc_curve(y_tr, ysc_tr)
                roc_parts.append(('ROC - Treino' , fpr_tr, tpr_tr, auc(fpr_tr, tpr_tr)))
                plotted_any = True
            else:
                res_tr = _macro_roc_from_df(_train_df, 'y_train')
                if res_tr is not None:
                    fpr_tr, tpr_tr, auc_tr = res_tr
                    roc_parts.append(('ROC (macro) - Treino', fpr_tr, tpr_tr, auc_tr))
                    plotted_any = True
        elif len(fprs_train) > 0 and K == 2:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr); tpr_i[0] = 0.0; tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            roc_parts.append(('ROC - Treino', mean_fpr, mean_tpr, np.mean(aurocs_train) if len(aurocs_train)>0 else auc(mean_fpr, mean_tpr)))
            plotted_any = True

        # Validação (novo em 8/1/1)
        if have_val:
            if K == 2 and 'prob_1' in _val_df.columns:
                y_vl, ysc_vl = _val_df['y_val'], _val_df['prob_1']
                fpr_vl, tpr_vl, _ = roc_curve(y_vl, ysc_vl)
                roc_parts.append(('ROC - Validação', fpr_vl, tpr_vl, auc(fpr_vl, tpr_vl)))
                plotted_any = True
            else:
                res_vl = _macro_roc_from_df(_val_df, 'y_val')
                if res_vl is not None:
                    fpr_vl, tpr_vl, auc_vl = res_vl
                    roc_parts.append(('ROC (macro) - Validação', fpr_vl, tpr_vl, auc_vl))
                    plotted_any = True

        # Teste
        if have_test:
            if K == 2 and 'prob_1' in _test_df.columns:
                y_ts, ysc_ts = _test_df['y_test'], _test_df['prob_1']
                fpr_ts, tpr_ts, _ = roc_curve(y_ts, ysc_ts)
                roc_parts.append(('ROC - Teste'  , fpr_ts, tpr_ts, auc(fpr_ts, tpr_ts)))
                plotted_any = True
            else:
                res_ts = _macro_roc_from_df(_test_df, 'y_test')
                if res_ts is not None:
                    fpr_ts, tpr_ts, auc_ts = res_ts
                    roc_parts.append(('ROC (macro) - Teste', fpr_ts, tpr_ts, auc_ts))
                    plotted_any = True
        elif len(fprs_test) > 0 and K == 2:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr); tpr_i[0] = 0.0; tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            roc_parts.append(('ROC - Teste', mean_fpr, mean_tpr, np.mean(aurocs_test) if len(aurocs_test)>0 else auc(mean_fpr, mean_tpr)))
            plotted_any = True

        if plotted_any and len(roc_parts) > 0:
            # 2 painéis (sem validação) ou 3 painéis (com validação)
            ncols = len(roc_parts)
            fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5), squeeze=False)
            axes = axes[0]
            for j, (title, fpr_j, tpr_j, auc_j) in enumerate(roc_parts):
                ax = axes[j]
                ax.plot(fpr_j, tpr_j, label=f'AUC = {auc_j:.4f}', lw=2)
                ax.plot([0, 1], [0, 1], 'k--', lw=1)
                ax.set_title(title)
                ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
                ax.legend(loc='lower right')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_val_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # ---------------- Matriz de Confusão (Treino/Validação/Teste) ----------------
    try:
        def build_cm_from_df(df_block, true_col, pred_col, labels):
            y_true = df_block[true_col].astype(int)
            y_pred = df_block[pred_col].astype(int)
            cm = confusion_matrix(y_true, y_pred, labels=labels)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        _train_df = globals().get('train_df', pd.DataFrame())
        _val_df   = globals().get('val_df',   pd.DataFrame())
        _test_df  = globals().get('test_df',  pd.DataFrame())

        cm_list = []  # lista de (title, cm, cm_pct)

        # Treino
        if isinstance(_train_df, pd.DataFrame) and not _train_df.empty and 'y_train' in _train_df.columns and 'y_pred' in _train_df.columns:
            cm_tr, cm_tr_pct = build_cm_from_df(_train_df, 'y_train', 'y_pred', class_labels)
            cm_list.append(('Matriz de Confusão - Treino', cm_tr, cm_tr_pct))
        elif K == 2 and 'TP' in df_resultados.columns:
            tr = df_resultados[df_resultados['Conjunto']=='Treino']
            TP = int(tr['TP'].sum()) if not tr.empty else 0
            FP = int(tr['FP'].sum()) if not tr.empty else 0
            TN = int(tr['TN'].sum()) if not tr.empty else 0
            FN = int(tr['FN'].sum()) if not tr.empty else 0
            cm_tr = np.array([[TN, FP], [FN, TP]])
            total = cm_tr.sum()
            cm_tr_pct = cm_tr / total * 100 if total>0 else np.zeros_like(cm_tr, float)
            cm_list.append(('Matriz de Confusão - Treino', cm_tr, cm_tr_pct))

        # Validação (novo em 8/1/1)
        if isinstance(_val_df, pd.DataFrame) and not _val_df.empty and 'y_val' in _val_df.columns and 'y_pred' in _val_df.columns:
            cm_vl, cm_vl_pct = build_cm_from_df(_val_df, 'y_val', 'y_pred', class_labels)
            cm_list.append(('Matriz de Confusão - Validação', cm_vl, cm_vl_pct))

        # Teste
        if isinstance(_test_df, pd.DataFrame) and not _test_df.empty and 'y_test' in _test_df.columns and 'y_pred' in _test_df.columns:
            cm_ts, cm_ts_pct = build_cm_from_df(_test_df, 'y_test', 'y_pred', class_labels)
            cm_list.append(('Matriz de Confusão - Teste', cm_ts, cm_ts_pct))
        elif K == 2 and 'TP' in df_resultados.columns:
            ts = df_resultados[df_resultados['Conjunto']=='Teste']
            TP = int(ts['TP'].sum()) if not ts.empty else 0
            FP = int(ts['FP'].sum()) if not ts.empty else 0
            TN = int(ts['TN'].sum()) if not ts.empty else 0
            FN = int(ts['FN'].sum()) if not ts.empty else 0
            cm_ts = np.array([[TN, FP], [FN, TP]])
            total = cm_ts.sum()
            cm_ts_pct = cm_ts / total * 100 if total>0 else np.zeros_like(cm_ts, float)
            cm_list.append(('Matriz de Confusão - Teste', cm_ts, cm_ts_pct))

        if len(cm_list) == 0:
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            ncols = len(cm_list)
            fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5), squeeze=False)
            axes = axes[0]

            def plot_cm(ax, cm, cm_pct, title, labels):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center'); ax.axis('off'); return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':11})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels([str(x) for x in labels]); ax.set_yticklabels([str(x) for x in labels])
                ax.set_title(title)

            for j, (title, cm_j, cm_j_pct) in enumerate(cm_list):
                plot_cm(axes[j], cm_j, cm_j_pct, title, class_labels)

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_val_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # ---------------- Importâncias / Pareto / Variância / Violinos (inalterado) ----------------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%','', regex=True)
        cols = [c for c in df_imp.columns if c not in ['Conjunto','Fold']]
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]
        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)
            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)')
            ax.set_title('Importância média das features (porcentagem)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                png_path = IMAGES_OUT / f'feature_importances_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela de importâncias
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                table_height = max(4, 0.3 * len(table_df))
                fig, ax = plt.subplots(figsize=(8, table_height))
                ax.axis('off')
                table = ax.table(cellText=table_df.values,
                                 colLabels=table_df.columns,
                                 rowLabels=table_df.index,
                                 loc='center')
                table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.2)
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # Pareto
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8) * 100
                pos = np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                selected_features = pareto_df.iloc[:pos+1].index.tolist() if pos < len(pareto_df) else pareto_df.index.tolist()
                pareto_df.reset_index().rename(columns={'index':'feature'}).to_csv(CSV_OUT / f'pareto_importances_{int((PARETO_THRESHOLD*100))}.csv', index=False)
                pd.DataFrame({'feature': selected_features,
                              'importance_pct': [float(imp_desc[f]) for f in selected_features]}
                             ).to_csv(CSV_OUT / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.csv', index=False)
                try:
                    import json
                    with open(MODEL_DIR / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.json', 'w') as fh:
                        json.dump({'threshold': float(PARETO_THRESHOLD),
                                   'selected_features': selected_features,
                                   'importances': {f: float(imp_desc[f]) for f in selected_features}}, fh, indent=2)
                except Exception:
                    pass
                # gráfico Pareto
                try:
                    fig, ax1 = plt.subplots(figsize=(10, 6))
                    x = pareto_df.index.astype(str)
                    ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                    ax1.set_ylabel('Importância (%)'); ax1.set_xticklabels(x, rotation=90)
                    ax2 = ax1.twinx()
                    ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                    ax2.axhline(threshold_pct, color='gray', linestyle='--')
                    ax2.set_ylabel('Cumulative (%)')
                    ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(selected_features)}')
                    plt.tight_layout()
                    try:
                        png_path = IMAGES_OUT / f'pareto_{int((PARETO_THRESHOLD*100))}.png'
                        fig.savefig(png_path, dpi=300, bbox_inches='tight')
                    except Exception:
                        pass
                    pdf.savefig(fig, bbox_inches='tight'); plt.close()
                except Exception as e_plot:
                    print('Não foi possível gerar gráfico Pareto:', e_plot)
            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)

            # Variance table: Treino vs Teste (mantido)
            try:
                if 'FEATURES' in globals() and not (train_df.empty or test_df.empty):
                    var_feats = [f for f in FEATURES if f in train_df.columns and f in test_df.columns]
                    num_feats = _numeric_common_features(train_df, test_df, var_feats)
                    if len(num_feats) == 0:
                        print('Nenhuma feature numérica comum/convertível em train/test para calcular variância.')
                    else:
                        tr_num = _coerce_numeric_df(train_df, num_feats)
                        te_num = _coerce_numeric_df(test_df,  num_feats)
                        common_num = [c for c in tr_num.columns if c in te_num.columns]
                        if len(common_num) == 0:
                            print('Após coerção, nenhuma feature numérica comum restou para variância.')
                        else:
                            var_train = tr_num[common_num].var(ddof=1)
                            var_test  = te_num[common_num].var(ddof=1)
                            var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                            var_df['diff'] = var_df['var_train'] - var_df['var_test']
                            var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100
                            try:
                                thresh_pct = float(VAR_DIFF_PCT_THRESHOLD) * 100
                            except Exception:
                                thresh_pct = 10.0
                            var_df['high_low'] = var_df['pct_diff'].apply(lambda x: 'High' if (not np.isnan(x) and x >= thresh_pct) else 'Low')
                            var_df = var_df.sort_values('var_train', ascending=False)
                            out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                            var_df.reset_index().rename(columns={'index': 'feature'}).to_csv(out_var_csv, index=False)
                            fig_h = min(12, 0.25 * len(var_df) + 2)
                            fig, ax = plt.subplots(figsize=(11.69, fig_h))
                            ax.axis('off')
                            tbl = ax.table(cellText=var_df.round(4).reset_index().values,
                                           colLabels=['feature', 'var_train', 'var_test', 'diff', 'pct_diff', 'high_low'],
                                           loc='center')
                            tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.2)
                            ax.set_title('Tabela: Variância das Features (Treino vs Teste)')
                            plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
                else:
                    print('Dados de treino/teste ou FEATURES ausentes; não foi possível calcular variâncias.')
            except Exception as e_var:
                print('Erro ao calcular tabela de variâncias:', e_var)

            # Violinos (Pareto) — mantido
            try:
                try:
                    _tmp = pd.read_csv(CSV_OUT / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.csv')
                    selected_features = _tmp['feature'].tolist()
                except Exception:
                    selected_features = []
                if not selected_features:
                    print('Nenhuma feature selecionada pelo Pareto para gerar violinos.')
                elif train_df.empty or test_df.empty:
                    print('Dados de treino ou teste ausentes para gerar violinos.')
                else:
                    vio_feats = _numeric_common_features(train_df, test_df, selected_features)
                    if len(vio_feats) == 0:
                        print('Nenhuma das features selecionadas (Pareto) é numérica/convertível em ambos os conjuntos.')
                    else:
                        tr_num = _coerce_numeric_df(train_df, vio_feats)
                        te_num = _coerce_numeric_df(test_df,  vio_feats)
                        common_vio = [c for c in vio_feats if c in tr_num.columns and c in te_num.columns]
                        if len(common_vio) == 0:
                            print('Após coerção, nenhuma feature numérica comum restou para violinos.')
                        else:
                            train_sub = tr_num[common_vio].copy(); train_sub['__set__'] = 'train'
                            test_sub  = te_num[common_vio].copy();  test_sub['__set__']  = 'test'
                            combined  = pd.concat([train_sub, test_sub], ignore_index=True)

                            per_page = 6
                            nplots = len(common_vio)
                            pages = int(np.ceil(nplots / per_page))
                            for p in range(pages):
                                start = p * per_page; end = min(start + per_page, nplots)
                                page_feats = common_vio[start:end]
                                fig, axes = plt.subplots(3, 2, figsize=(8.27, 11.69))  # A4 portrait
                                axes = axes.flatten()
                                for i, feat in enumerate(page_feats):
                                    ax = axes[i]
                                    sns.violinplot(x='__set__', y=feat, data=combined,
                                                   order=['train', 'test'],
                                                   ax=ax, inner='quartile',
                                                   palette=['#4c72b0', '#55a868'])
                                    ax.set_title(f'Violin: {feat} (train vs test)')
                                    ax.set_xlabel('')
                                for j in range(len(page_feats), len(axes)):
                                    axes[j].axis('off')
                                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight')

                                # imagens individuais
                                try:
                                    for feat in page_feats:
                                        fig_f, ax_f = plt.subplots(figsize=(6,4))
                                        sns.violinplot(x='__set__', y=feat, data=combined,
                                                       order=['train', 'test'],
                                                       ax=ax_f, inner='quartile',
                                                       palette=['#4c72b0', '#55a868'])
                                        ax_f.set_title(f'Violin: {feat} (train vs test)')
                                        ax_f.set_xlabel('')
                                        plt.tight_layout()
                                        png_path = IMAGES_OUT / f'violin_{feat}.png'
                                        fig_f.savefig(png_path, dpi=300, bbox_inches='tight')
                                        plt.close(fig_f)
                                except Exception:
                                    pass
                                plt.close(fig)
            except Exception as e_violin:
                print('Não foi possível gerar gráficos de violino:', e_violin)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias/Pareto/variância/violinos:', e)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# === XGBoost Básico — agora com split 9/1 (train/test) ===
try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

# Para 9/1 vamos usar 10% teste e 90% treino
TEST_SIZE = 0.10

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
import pandas as pd
import joblib
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

# ---------- sanity ----------
if 'FEATURES' not in globals():
    raise RuntimeError('FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('Dataframe df não encontrado — execute as células anteriores para carregar os dados')

# ---------- prepara X e y ----------
X = df[FEATURES].copy()
y = df['y'].copy().astype(int)  # garante inteiro

# detecta nº de classes
K = int(y.nunique())
class_labels = sorted(y.unique().tolist())

# ajusta params do XGB para multi-classe, se necessário
xgb_params_basic = dict(xgb_params)  # cópia segura
if K > 2:
    xgb_params_basic['objective']   = 'multi:softprob'
    xgb_params_basic['eval_metric'] = xgb_params_basic.get('eval_metric', 'mlogloss')
    xgb_params_basic['num_class']   = K
else:
    xgb_params_basic['objective']   = xgb_params_basic.get('objective', 'binary:logistic')
    xgb_params_basic['eval_metric'] = xgb_params_basic.get('eval_metric', 'logloss')

# habilita categóricas no XGBoost
xgb_params_basic['enable_categorical'] = True

# ---------- split 9/1 mantendo índice original ----------
# 1) separa 10% para TESTE (não haverá conjunto de validação separado)
X_train_tmp, X_test_tmp, y_train_tmp, y_test_tmp, idx_train_tmp, idx_test_tmp = train_test_split(
    X.reset_index(),               # X com coluna 'index' (original)
    y.reset_index(),               # y com coluna 'index'
    df.reset_index()['index'],     # índices originais do df
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RT_RANDOM_STATE
)

def _restore_index(Xblock):
    if isinstance(Xblock, pd.DataFrame) and 'index' in Xblock.columns:
        orig_idx = Xblock['index'].values
        return Xblock.drop(columns=['index']).set_index(pd.Index(orig_idx))
    return Xblock

# restaura índices de treino e teste
X_train = _restore_index(X_train_tmp)
X_test  = _restore_index(X_test_tmp)

# y_* vieram com col 'index' e 'y'
def _restore_y(yblock):
    if hasattr(yblock, 'columns') and 'y' in yblock.columns:
        return yblock.set_index(pd.Index(yblock['index'].values))['y'].astype(int)
    return yblock.astype(int)

y_train = _restore_y(y_train_tmp)
y_test  = _restore_y(y_test_tmp)

# ---------- harmoniza categóricas em 2 conjuntos ----------
def _harmonize_categoricals_2(df_tr: pd.DataFrame, df_te: pd.DataFrame):
    tr, te = df_tr.copy(), df_te.copy()
    all_cols = tr.columns
    cat_cols = []
    for c in all_cols:
        if (tr[c].dtype == 'object') or (te[c].dtype == 'object') \
           or pd.api.types.is_categorical_dtype(tr[c]) or pd.api.types.is_categorical_dtype(te[c]):
            cat_cols.append(c)
    for c in cat_cols:
        tr[c] = tr[c].astype(str)
        te[c] = te[c].astype(str)
        cats = pd.Index(tr[c].unique()).union(pd.Index(te[c].unique()))
        tr[c] = pd.Categorical(tr[c], categories=cats)
        te[c] = pd.Categorical(te[c], categories=cats)
    return tr, te, cat_cols

X_train, X_test, CAT_COLS = _harmonize_categoricals_2(X_train, X_test)

# ---------- treina ----------
model_basic = XGBClassifier(**xgb_params_basic)
model_basic.fit(X_train, y_train)

# ---------- predições ----------
y_pred_train = model_basic.predict(X_train)
y_pred_test  = model_basic.predict(X_test)

proba_train = model_basic.predict_proba(X_train)
proba_test  = model_basic.predict_proba(X_test)

if K == 2:
    y_proba_train_pos = proba_train[:, 1]
    y_proba_test_pos  = proba_test[:, 1]
else:
    y_proba_train_pos = y_proba_test_pos = None

# ---------- métricas ----------
acc_train = accuracy_score(y_train, y_pred_train)
acc_test  = accuracy_score(y_test,  y_pred_test)

if K == 2:
    f1_tr   = f1_score(y_train, y_pred_train)
    f1_ts   = f1_score(y_test,  y_pred_test)
    prec_tr = precision_score(y_train, y_pred_train)
    prec_ts = precision_score(y_test,  y_pred_test)
    rec_tr  = recall_score(y_train, y_pred_train)
    rec_ts  = recall_score(y_test,  y_pred_test)
    auc_tr  = roc_auc_score(y_train, y_proba_train_pos) if len(np.unique(y_train))>1 else np.nan
    auc_ts  = roc_auc_score(y_test,  y_proba_test_pos)  if len(np.unique(y_test))>1  else np.nan
else:
    f1_tr   = f1_score(y_train, y_pred_train, average='macro')
    f1_ts   = f1_score(y_test,  y_pred_test,  average='macro')
    prec_tr = precision_score(y_train, y_pred_train, average='macro')
    prec_ts = precision_score(y_test,  y_pred_test,  average='macro')
    rec_tr  = recall_score(y_train, y_pred_train, average='macro')
    rec_ts  = recall_score(y_test,  y_pred_test,  average='macro')
    try:
        auc_tr = roc_auc_score(y_train, proba_train, multi_class='ovr', average='macro')
    except Exception:
        auc_tr = np.nan
    try:
        auc_ts = roc_auc_score(y_test,  proba_test,  multi_class='ovr', average='macro')
    except Exception:
        auc_ts = np.nan

print(f"[XGB Básico 9/1] K={K} | Acc train: {acc_train:.4f} | test: {acc_test:.4f}")

# ---------- salva modelo e artefatos ----------
model_name = MODEL_DIR / f'xgb_basic_811_rs{RT_RANDOM_STATE}.pkl'  # mantém nome pra compatibilidade
joblib.dump(model_basic, model_name)

BASICO_CSV.mkdir(parents=True, exist_ok=True)

# features
features = X.columns.tolist()
joblib.dump(features, MODEL_DIR / f'features_xgb_basic_811_rs{RT_RANDOM_STATE}.pkl')

# ===================== SOMENTE DOIS ARQUIVOS X_* BÁSICOS =====================
X_train.to_csv(BASICO_CSV / f'X_train_xgb_basic_811_rs{RT_RANDOM_STATE}.csv')
X_test.to_csv( BASICO_CSV / f'X_test_xgb_basic_811_rs{RT_RANDOM_STATE}.csv')
# ==========================================================

# y em CSV (se ainda quiser manter)
pd.Series(y_train, name='y_train').to_csv(BASICO_CSV / f'y_train_xgb_basic_811_rs{RT_RANDOM_STATE}.csv', index=True)
pd.Series(y_test,  name='y_test').to_csv( BASICO_CSV / f'y_test_xgb_basic_811_rs{RT_RANDOM_STATE}.csv',  index=True)

# ---------- DataFrames com probabilidades + y_pred ----------
def _proba_df(proba_mat: np.ndarray, index_like, prefix='prob_'):
    if proba_mat.ndim == 1:
        proba_mat = np.vstack([1.0 - proba_mat, proba_mat]).T
    k_local = proba_mat.shape[1]
    cols = [f'{prefix}{i}' for i in range(k_local)]
    return pd.DataFrame(proba_mat, index=index_like, columns=cols)

proba_train_df = _proba_df(proba_train, X_train.index)
proba_test_df  = _proba_df(proba_test,  X_test.index)

out_train = X_train.copy(); out_train['y_train'] = y_train; out_train['y_pred'] = y_pred_train
out_train = out_train.merge(proba_train_df, left_index=True, right_index=True, how='left')

out_test = X_test.copy(); out_test['y_test'] = y_test; out_test['y_pred'] = y_pred_test
out_test = out_test.merge(proba_test_df, left_index=True, right_index=True, how='left')

if K == 2:
    out_train['y_proba'] = out_train['prob_1']
    out_test['y_proba']  = out_test['prob_1']
else:
    out_test['y_proba_pred'] = proba_test[np.arange(len(out_test)), y_pred_test]

out_train.to_csv(BASICO_CSV / f'xgb_train_with_proba_basic_811_rs{RT_RANDOM_STATE}.csv')
out_test.to_csv( BASICO_CSV / f'xgb_test_with_proba_basic_811_rs{RT_RANDOM_STATE}.csv')

# ---------- métricas -> CSV (sem validação) ----------
metrics_basic_cols = {
    'model': str(model_name.name),
    'K_classes': K,
    'split': '9/1',
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train,
    'acc_val':   np.nan,        # mantidos por compatibilidade
    'acc_test':  acc_test,
    ('f1_train_macro' if K>2 else 'f1_train'): f1_tr,
    ('f1_val_macro'   if K>2 else 'f1_val'):   np.nan,
    ('f1_test_macro'  if K>2 else 'f1_test'):  f1_ts,
    ('precision_train_macro' if K>2 else 'precision_train'): prec_tr,
    ('precision_val_macro'   if K>2 else 'precision_val'):   np.nan,
    ('precision_test_macro'  if K>2 else 'precision_test'):  prec_ts,
    ('recall_train_macro' if K>2 else 'recall_train'): rec_tr,
    ('recall_val_macro'   if K>2 else 'recall_val'):   np.nan,
    ('recall_test_macro'  if K>2 else 'recall_test'):  rec_ts,
    ('auc_train_macro_ovr' if K>2 else 'auc_train'): auc_tr,
    ('auc_val_macro_ovr'   if K>2 else 'auc_val'):   np.nan,
    ('auc_test_macro_ovr'  if K>2 else 'auc_test'):  auc_ts
}
pd.DataFrame([metrics_basic_cols]).to_csv(BASICO_CSV / f'xgb_model_basic_811.csv', index=False)

# ---------- dataset augmentado (original + preds do TESTE) ----------
try:
    df_orig = df.reset_index().rename(columns={'index': 'orig_index'})
    test_with_idx = out_test.reset_index().rename(columns={'index':'orig_index'})
    prob_cols = [c for c in test_with_idx.columns if c.startswith('prob_')]
    keep_cols = ['orig_index', 'y_pred'] + (['y_proba'] if K==2 else ['y_proba_pred']) + prob_cols
    preds_df = test_with_idx[keep_cols].copy()
    df_augmented = pd.merge(df_orig, preds_df, how='left', on='orig_index')
    df_augmented.to_csv(BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_811_rs{RT_RANDOM_STATE}.csv', index=False)
    df_augmented.to_csv(CSV_OUT   / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_811_rs{RT_RANDOM_STATE}.csv', index=False)
except Exception as e_aug:
    print('Não foi possível gerar dataset augmentado do básico:', e_aug)

# ---------- amostra aleatória de índices (agora usando X_train) ----------
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_train.index, size=min(num_instancias, len(X_train)), replace=False)

print('Modelo salvo em:', model_name)
print('Métricas (9/1) salvas em:', BASICO_CSV / f'xgb_model_basic_811.csv')
print('Train/Test com probabilidades salvos em BASICO_CSV.')
print('Dataset augmentado (TESTE) salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_811_rs{RT_RANDOM_STATE}.csv')
print('Índices aleatórios (train):', indices_aleatorios)

In [ ]:
# Salvar X_*_basic_full.csv (apenas TRAIN e TEST no cenário 9/1)
# com todas as colunas do arquivo cru + y, y_pred, y_proba (+ prob_0..prob_{K-1} quando K>2)

import numpy as np
import pandas as pd

# --- carrega CSV cru original ---
raw_df = pd.read_csv(DATASET_PATH)

# --- checagens básicas ---
if 'df' not in globals():
    raise RuntimeError("DataFrame 'df' não encontrado. Execute as células anteriores.")
if 'y' not in df.columns:
    raise RuntimeError("Coluna 'y' não encontrada em df. Execute as células anteriores.")
K = int(df['y'].nunique())

def _resolve_block_refs(split: str):
    """
    Retorna tupla (X_blk_name, y_true_varname, y_pred_varnames, proba_varnames, out_df_name)
    sem acessar ainda os valores (deixa para _resolve_preds_and_proba).
    """
    if split == 'train':
        return (
            'X_train',                 # X_blk_name
            'y_train',                 # y_true_varname
            ['y_pred_train', 'y_pred_train_bin'],  # possíves nomes de y_pred
            ['proba_train', 'y_proba_train', 'proba_train_all', 'y_proba_train_pos'],  # proba
            'out_train'               # out_df_name (fallback)
        )
    # default: 'test'
    return (
        'X_test',
        'y_test',
        ['y_pred_test', 'y_pred_test_bin'],
        ['proba_test', 'y_proba_test', 'proba_test_all', 'y_proba_test_pos'],
        'out_test'
    )

def _resolve_preds_and_proba(split: str):
    """
    split: 'train' | 'test'
    Retorna (X_blk_index, y_true, y_pred, proba_mat) alinhados ao índice de X_<split>.
    Usa em ordem:
      1) variáveis diretas (X_*, y_*, y_pred_*, proba_*)
      2) fallback em out_* (merge do bloco básico)
    """
    X_name, y_name, ypred_names, proba_names, out_name = _resolve_block_refs(split)

    # X_*
    if X_name not in globals() or globals()[X_name] is None or len(globals()[X_name]) == 0:
        raise RuntimeError(f"{X_name} não encontrado ou vazio — nada a salvar para '{split}'.")
    X_blk = globals()[X_name]

    # y_*
    if y_name not in globals():
        raise RuntimeError(f"{y_name} não encontrado para '{split}'.")
    y_true = globals()[y_name]

    # y_pred_* (primeiro que existir)
    y_pred = None
    for nm in ypred_names:
        if nm in globals():
            y_pred = globals()[nm]
            break

    # proba_* (primeiro que existir)
    proba = None
    for nm in proba_names:
        if nm in globals():
            proba = globals()[nm]
            break

    # --- fallback: tenta em out_* se necessário ---
    if (y_pred is None) or (proba is None):
        if out_name in globals():
            try:
                out_df = globals()[out_name]
                # y_true pode estar em out_df como y_<split>
                col_true = f'y_{split}'
                if isinstance(out_df, pd.DataFrame) and col_true in out_df.columns:
                    y_true = out_df[col_true]
                # y_pred
                if y_pred is None and 'y_pred' in out_df.columns:
                    y_pred = out_df['y_pred'].values
                # prob_*
                if proba is None:
                    prob_cols = [c for c in out_df.columns if str(c).startswith('prob_')]
                    if len(prob_cols) > 0:
                        proba = out_df[prob_cols].to_numpy()
            except Exception:
                pass

    if proba is None:
        raise RuntimeError(f"Probabilidades não disponíveis para '{split}'.")

    # normaliza proba: se 1D (binário), vira (n,2)
    proba = np.asarray(proba)
    if proba.ndim == 1:
        proba = np.vstack([1.0 - proba, proba]).T

    # garante alinhamento pelos índices de X_*
    if hasattr(y_true, 'reindex'):
        y_true = y_true.reindex(X_blk.index)
    y_pred = np.asarray(y_pred)
    if y_pred.shape[0] != len(X_blk):
        raise RuntimeError(f"Tamanho de y_pred({y_pred.shape[0]}) != {X_name}({len(X_blk)})")
    if proba.shape[0] != len(X_blk):
        raise RuntimeError(f"Tamanho de proba({proba.shape[0]}) != {X_name}({len(X_blk)})")

    return X_blk.index, y_true, y_pred, proba

def _save_full_split(indices, y_true, y_pred, proba_mat, split_name):
    # Seleciona linhas do CSV cru pelo índice original
    df_full = raw_df.loc[indices].copy()

    # y (verdadeiro) e y_pred
    y_true_arr = pd.Series(y_true, index=indices).values
    y_pred_arr = pd.Series(y_pred, index=indices).values
    df_full['y'] = y_true_arr
    df_full['y_pred'] = y_pred_arr

    # Probabilidades por classe
    k_local = proba_mat.shape[1]
    for j in range(k_local):
        df_full[f'prob_{j}'] = proba_mat[:, j]

    # y_proba: binário -> prob_1 ; multi -> prob da classe predita
    if k_local == 2:
        df_full['y_proba'] = proba_mat[:, 1]
    else:
        df_full['y_proba'] = proba_mat[np.arange(len(df_full)), df_full['y_pred'].astype(int)]

    # Salva
    df_full.to_csv(BASICO_CSV / f'X_{split_name}_basic_full.csv', index=True)

# --- TEST ---
try:
    idx, y_true_test, y_pred_test_use, proba_test_use = _resolve_preds_and_proba('test')
    _save_full_split(idx, y_true_test, y_pred_test_use, proba_test_use, 'test')
    print("✅ X_test_basic_full.csv salvo.")
except Exception as e:
    print("⚠️  Teste não salvo:", e)

# --- TRAIN ---
try:
    idx, y_true_train, y_pred_train_use, proba_train_use = _resolve_preds_and_proba('train')
    _save_full_split(idx, y_true_train, y_pred_train_use, proba_train_use, 'train')
    print("✅ X_train_basic_full.csv salvo.")
except Exception as e:
    print("⚠️  Treino não salvo:", e)

print("📁 Arquivos gerados em:", BASICO_CSV)
